In [1]:
!pip install -q transformers datasets accelerate wandb peft

In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
WANDB_KEY = user_secrets.get_secret("WANDB_API_KEY")

In [4]:
import wandb 

wandb.login(key=WANDB_KEY)
run = wandb.init(
        project="24f2008341-t22026",
        name="exp4 " + "dberta-v3-base",
        tags=["rag", "retrieval", "deberta-v3-base", "multiple-choice", "M3", "model-5"],
    )

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


In [5]:
import os
import numpy as np
import pandas as pd
import torch
import wandb
from dataclasses import dataclass
from typing import Optional, Union
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForMultipleChoice, 
    TrainingArguments, 
    Trainer,
    PreTrainedTokenizerBase
)
from transformers.tokenization_utils_base import PaddingStrategy

In [55]:
MODEL_NAME = "microsoft/deberta-v3-base"  # Try "microsoft/deberta-v3-large" for higher scores
MAX_LEN = 256
BATCH_SIZE = 8       # Per GPU batch size (Effective = 4 * 2 GPUs = 8)
EPOCHS = 5
LEARNING_RATE = 2e-4

OPTION_MAP = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
REV_OPTION_MAP = {v: k for k, v in OPTION_MAP.items()}
OPTIONS = ['A', 'B', 'C', 'D', 'E']

In [56]:
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

# Encode target letters into integers 0..4
train_df['label'] = train_df['answer'].map(OPTION_MAP)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [57]:
def preprocess_function(examples):
    first_sentences = [[prompt] * 5 for prompt in examples['prompt']]
    second_sentences = [
        [examples[option][i] for option in OPTIONS] 
        for i in range(len(examples['prompt']))
    ]
    
    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])
    
    tokenized_inputs = tokenizer(
        first_sentences, 
        second_sentences, 
        truncation=True, 
        max_length=MAX_LEN
    )
    
    # Reshape tokenized inputs to (batch_size, 5, seq_len)
    batch_outputs = {
        k: [v[i : i + 5] for i in range(0, len(v), 5)] 
        for k, v in tokenized_inputs.items()
    }
    
    # Preserve the target label for loss calculation
    if 'label' in examples:
        batch_outputs['label'] = examples['label']
        
    return batch_outputs

In [58]:
train_ds = Dataset.from_pandas(train_df)
test_ds = Dataset.from_pandas(test_df)

# Split train set for local evaluation
train_val_ds = train_ds.train_test_split(test_size=0.15, seed=42)

# Determine raw text columns to remove (keeping target labels intact)
train_cols_to_remove = [col for col in train_df.columns if col in train_ds.column_names and col != 'label']
test_cols_to_remove = [col for col in test_df.columns if col in test_ds.column_names]

encoded_train = train_val_ds['train'].map(
    preprocess_function, 
    batched=True, 
    remove_columns=train_cols_to_remove
)
encoded_val = train_val_ds['test'].map(
    preprocess_function, 
    batched=True, 
    remove_columns=train_cols_to_remove
)
encoded_test = test_ds.map(
    preprocess_function, 
    batched=True, 
    remove_columns=test_cols_to_remove
)

Map:   0%|          | 0/1700 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [59]:
@dataclass
class DataCollatorForMultipleChoice:
    tokenizer: PreTrainedTokenizerBase
    padding: Union[bool, str, PaddingStrategy] = True
    max_length: Optional[int] = None
    pad_to_multiple_of: Optional[int] = None

    def __call__(self, features):
        label_name = "label" if "label" in features[0] else "labels"
        labels = [feature.pop(label_name) for feature in features] if label_name in features[0] else None
        
        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])
        
        flattened_features = [
            [{k: v[i] for k, v in feature.items()} for i in range(num_choices)]
            for feature in features
        ]
        flattened_features = sum(flattened_features, [])
        
        batch = self.tokenizer.pad(
            flattened_features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )
        
        batch = {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}
        if labels is not None:
            batch["labels"] = torch.tensor(labels, dtype=torch.long)
        return batch

In [60]:
def compute_map3(eval_predictions):
    logits, labels = eval_predictions
    top3_preds = np.argsort(-logits, axis=1)[:, :3]
    
    scores = []
    for preds, target in zip(top3_preds, labels):
        if target in preds:
            rank = np.where(preds == target)[0][0] + 1
            scores.append(1.0 / rank)
        else:
            scores.append(0.0)
            
    return {"map@3": np.mean(scores)}

In [61]:
model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    fp16=True,                             # Mixed precision for T4 GPUs
    max_grad_norm=0.0,
    report_to="wandb",                     # Enable W&B Logging
    load_best_model_at_end=True,
    metric_for_best_model="map@3",
    greater_is_better=True,
    logging_steps=50,
)

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight                

In [62]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_train,
    eval_dataset=encoded_val,
    processing_class=tokenizer,
    data_collator=DataCollatorForMultipleChoice(tokenizer=tokenizer),
    compute_metrics=compute_map3,
)

trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Map@3
1,0.000000,3.218750,0.408889
2,0.000000,3.218750,0.408889
3,0.000000,3.218750,0.408889


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deberta.encoder.layer.0.output.LayerNorm.bias', 'deberta.encoder.layer.1.attention.output.LayerNorm.weight', 'deberta.encoder.layer.1.attention.output.LayerNorm.bias', 'deberta.encoder.layer.1.output.LayerNorm.weight', 'deberta.encoder.layer.1.output.LayerNorm.bias', 'deberta.encoder.layer.2.attention.output.LayerNorm.weight', 'deberta.encoder.layer.2.attention.output.LayerNorm.bias', 'deberta.encoder.layer.2.output.LayerNorm.weight', 'deberta.encoder.layer.2.output.LayerNorm.bias', 'deberta.encoder.layer.3.attention.output.LayerNorm.weight', 'deberta.encoder.layer.3.attention.output.LayerNorm.bias', 'deberta.encoder.layer.3.output.LayerNorm.weight', 'deberta.encoder.layer.3.output.Laye

TrainOutput(global_step=639, training_loss=0.24708346992218067, metrics={'train_runtime': 158.5834, 'train_samples_per_second': 32.16, 'train_steps_per_second': 4.029, 'total_flos': 1266089403101040.0, 'train_loss': 0.24708346992218067, 'epoch': 3.0})

In [63]:
raw_predictions = trainer.predict(encoded_test)
test_logits = raw_predictions.predictions

# Get top 3 predictions for each test instance
top3_indices = np.argsort(-test_logits, axis=1)[:, :3]

formatted_preds = []
for idx_row in top3_indices:
    top3_letters = [REV_OPTION_MAP[idx] for idx in idx_row]
    formatted_preds.append(" ".join(top3_letters))

# Build submission DataFrame matching sample_submission.csv format
submission_df = pd.DataFrame({
    'id': test_df['id'],
    'Prediction': formatted_preds
})

submission_df.to_csv('submission.csv', index=False)
print("Submission generated successfully!")
print(submission_df.head())

Submission generated successfully!
   id Prediction
0   1      A B C
1   2      D A B
2   3      D A B
3   4      A C B
4   5      A B C


In [64]:
wandb.finish()

eval/loss,▁▁▁
eval/map@3,▁▁▁
eval/runtime,█▁▁
eval/samples_per_second,▁██
eval/steps_per_second,▁██
test/runtime,▁
test/samples_per_second,▁
test/steps_per_second,▁
train/epoch,▁▂▂▃▃▃▄▅▅▅▆▆▇███
train/global_step,▁▂▂▃▃▃▄▅▅▅▆▆▇████
+2,...
